# 1. Rank donors by major-gift propensity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PhilanthroPy-Project/PhilanthroPy/blob/main/examples/notebooks/01_quickstart_propensity.ipynb)

Fit a propensity model on a synthetic donor pool, score a held-out set, and read
the result honestly. About a minute end to end.

What you get by the last cell: a ranked prospect list, the held-out ROC-AUC that
ranking is worth, which signals moved the score, and a picture of how much the
two groups overlap.

Everything here is synthetic. It demonstrates the workflow, not a number to
quote for your own program.

In [ ]:
# Colab and other fresh environments only; a local checkout already has it.
try:
    import philanthropy
except ImportError:
    !pip install -q "philanthropy[viz]"
    import philanthropy

print("philanthropy", philanthropy.__version__)

## Fit and score

Split before fitting. Scoring the rows you trained on tells you nothing: a
random forest's leaves go pure, so it recites its training set and reports a
gap that will not survive contact with next year's data.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from philanthropy.datasets import generate_synthetic_donor_data
from philanthropy.models import DonorPropensityModel

FEATURES = ["total_gift_amount", "years_active", "event_attendance_count"]

df = generate_synthetic_donor_data(n_samples=2000, random_state=42)
X = df[FEATURES].to_numpy()
y = df["is_major_donor"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = DonorPropensityModel(n_estimators=200, random_state=0)
model.fit(X_train, y_train)

scores = model.predict_affinity_score(X_test)   # 0-100, not a probability
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"held-out ROC-AUC: {auc:.3f}")

## The call list

`predict_affinity_score` returns a 0-100 rank, not a calibrated probability.
Sort by it and hand your officers the top of the list.

In [ ]:
prospects = pd.DataFrame(X_test, columns=FEATURES)
prospects["affinity_score"] = scores
prospects.sort_values("affinity_score", ascending=False).head(10)

## Read it honestly

Now look at the two distributions rather than the headline number.

In [ ]:
pd.Series(scores).groupby(y_test).describe()[["count", "mean", "min", "max"]]

The distributions **overlap**. Some non-major donors score in the nineties and
some major donors score near zero. Ranking works; clean separation does not
exist, and a list cut at any single threshold will contain mistakes.

Pick the threshold from your team's capacity, not from this table. If forty
visits is what the quarter allows, take the top forty and stop.

In [ ]:
from philanthropy.visualisation import plot_affinity_distribution

ax = plot_affinity_distribution(scores, labels=y_test)
ax.figure

## Which signals moved the score

Permutation importance, measured against the held-out split rather than the
training data, because attribution scored on data the model memorised tells you
about the memorisation.

In [ ]:
from philanthropy.inspection import donor_feature_importance

donor_feature_importance(
    model, X_test, y_test, feature_names=FEATURES, random_state=0
)

## Next

- **[2. Where the leakage actually is](02_temporal_leakage.ipynb)**: the same
  kind of model, measured properly across time. This is the one that matters.
- **[3. A grateful-patient pipeline](03_grateful_patient_pipeline.ipynb)**: the
  academic-medical-center path, with clinical encounters.

`total_gift_amount` carrying most of the signal above is a warning, not a
result: predicting "is a major donor" from cumulative lifetime giving is
close to predicting the label from itself. Notebook 2 is about exactly this
class of mistake.